<!--nav--> [🗺 Learning path](README.md) · **41/41** · ◀ [Anatomy of a Decode Step](./Anatomy_Of_A_Decode_Step.ipynb) · [🏁 back to the map](README.md)

# The Optimization Stack: Why Gains Don't Multiply

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/The_Optimization_Stack.ipynb)

Notebook 38 took **one decode step** apart. This one puts **all the optimizations** together — and
confronts the thing every vendor benchmark quietly hides:

> "AWQ gives 1.6×." "Speculation gives 2.5×." "FP8 KV gives 2× concurrency."
> **Therefore 8×, right?** No. Often barely 2×.

Optimizations compete for the *same* bottleneck. Fix the biggest slice and a different slice becomes
the ceiling — so the second optimization has less to remove than it did alone. Sometimes they
actively fight: two of the techniques in this track make each other **worse**, and one pair is
mutually destructive.

| Part | What you'll learn |
|---|---|
| **1** | The composition model — one step budget, transformed by each optimization |
| **2** | **The interaction matrix, computed** (not asserted) from that model |
| **3** | **Bottleneck migration**: watch the ceiling move as you stack |
| **4** | The stacking waterfall: actual vs the multiplication people expect |
| **5** | **Search**: the best stack under an accuracy-risk and effort budget |
| **6** | Interactions the arithmetic can't see (including one genuinely destructive pair) |
| **7** | Per-workload recipes and a one-page decision sheet |

**Runs on:** any CPU. Everything is derived from the step model of notebook 38.

In [ ]:
import math, json, random, uuid, statistics
from collections import defaultdict
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · The composition model

We reuse notebook 38's decode-step decomposition, then express each optimization as a
**transformation of that budget** rather than as a headline multiplier. That's the whole trick: if
optimizations are transformations of a shared state, composition is just function composition, and
the interactions fall out of the arithmetic instead of being guessed.

Two model details worth stating explicitly, because they're what make the interactions *real*:

- **Speculative decoding** doesn't make a step faster — it makes *fewer* steps. Verifying `k` draft
  tokens does `k×` the compute for the *same* weight read. So it converts spare compute into tokens,
  and when there is no spare compute it converts nothing.
- **Larger batch** does the same thing from the other direction: more tokens per weight-read, at more
  compute per step.

Both spend the same currency — **spare compute** — which is precisely why they interact.

In [ ]:
# The step budget from notebook 38, extended so optimizations can transform it.
HW = {
  "T4":        dict(bw=0.32e12, tf=65e12,   link=None,   launch_us=2.2, vram=16),
  "A100 80GB": dict(bw=2.04e12, tf=312e12,  link=300e9,  launch_us=1.5, vram=80),
  "H100 SXM":  dict(bw=3.35e12, tf=990e12,  link=900e9,  launch_us=1.2, vram=80),
  "MI300X":    dict(bw=5.30e12, tf=1307e12, link=800e9,  launch_us=1.5, vram=192),
}
MODELS = {
  "Qwen2.5-0.5B": dict(params=0.5e9, layers=24, hidden=896,  kv_heads=2, head_dim=64,  vocab=152000),
  "Llama-3.1-8B": dict(params=8e9,   layers=32, hidden=4096, kv_heads=8, head_dim=128, vocab=128000),
  "Llama-3.1-70B":dict(params=70e9,  layers=80, hidden=8192, kv_heads=8, head_dim=128, vocab=128000),
}
KERNELS_PER_LAYER = 14
BW_EFF, FLOP_EFF = 0.75, 0.60

def base_config(model="Llama-3.1-8B", gpu="H100 SXM", batch=32, ctx=2048):
    return dict(model=model, gpu=gpu, batch=batch, ctx=ctx,
                weight_bytes=2.0, kv_bytes=2.0, tp=1, graphs=True,
                spec_k=0, spec_alpha=0.0, kernel_eff=1.0, attn_ctx_frac=1.0)

def evaluate(cfg):
    m, g = MODELS[cfg["model"]], HW[cfg["gpu"]]
    bw = g["bw"] * BW_EFF * cfg["tp"]
    flops = g["tf"] * FLOP_EFF * cfg["tp"] * cfg["kernel_eff"]
    batch, k = cfg["batch"], cfg["spec_k"]

    # A verify pass covers (k+1) token positions per sequence: same weight read, more compute.
    positions = batch * (k + 1)
    gemm_mem = (m["params"] * cfg["weight_bytes"]) / bw
    gemm_cmp = (2 * m["params"] * positions) / flops
    gemm = max(gemm_mem, gemm_cmp)

    kv_per_tok = 2 * m["layers"] * m["kv_heads"] * m["head_dim"] * cfg["kv_bytes"]
    attention = (kv_per_tok * cfg["ctx"] * cfg["attn_ctx_frac"] * positions) / bw
    sampling = (positions * m["vocab"] * 4 * 3) / bw
    comm = 0.0
    if cfg["tp"] > 1 and g["link"]:
        comm = (2 * m["hidden"] * positions * 2 * (2 * (cfg["tp"] - 1) / cfg["tp"])
                * m["layers"] / g["link"])
    n_kernels = KERNELS_PER_LAYER * m["layers"]
    launch = (g["launch_us"] * 1e-6) * (3 if cfg["graphs"] else n_kernels)

    # A draft model costs a fraction of a target pass, k times over.
    draft = (k * 0.06 * gemm_mem) if k else 0.0

    parts = {"GEMMs": gemm, "attention": attention, "communication": comm,
             "sampling": sampling, "launch": launch, "draft": draft}
    step_s = sum(parts.values())

    # Tokens emitted per step: 1 normally; with speculation, the truncated geometric mean (nb 24).
    a = cfg["spec_alpha"]
    tokens_per_seq = ((1 - a ** (k + 1)) / (1 - a)) if k and a < 1 else 1.0
    tps = batch * tokens_per_seq / step_s

    return {"parts": parts, "step_ms": step_s * 1000, "tokens_per_s": tps,
            "tpot_ms": step_s * 1000 / tokens_per_seq,
            "bottleneck": max(parts, key=parts.get),
            "gemm_bound": "compute" if gemm_cmp > gemm_mem else "memory",
            "spare_compute": 1 - min(1.0, gemm_cmp / max(gemm_mem, 1e-12))}

# --- optimizations as transformations of that config --------------------------------------
def opt_int4(c):   c = dict(c); c["weight_bytes"] = 0.5; c["kernel_eff"] = 0.85; return c
def opt_fp8w(c):   c = dict(c); c["weight_bytes"] = 1.0; return c
def opt_fp8kv(c):  c = dict(c); c["kv_bytes"] = 1.0; return c
def opt_graphs(c): c = dict(c); c["graphs"] = True; return c
def opt_batch2(c): c = dict(c); c["batch"] = c["batch"] * 2; return c
def opt_spec(c):   c = dict(c); c["spec_k"] = 4; c["spec_alpha"] = 0.75; return c
def opt_tp2(c):    c = dict(c); c["tp"] = 2; return c
def opt_evict(c):  c = dict(c); c["attn_ctx_frac"] = 0.25; return c

OPTS = {
    "int4 weights":  opt_int4,
    "fp8 weights":   opt_fp8w,
    "fp8 KV cache":  opt_fp8kv,
    "2x batch":      opt_batch2,
    "speculation":   opt_spec,
    "TP=2":          opt_tp2,
    "KV eviction":   opt_evict,
}

base = base_config()
b = evaluate(base)
print(f"baseline: {base['model']} on {base['gpu']}, batch {base['batch']}, {base['ctx']} ctx")
print(f"  step {b['step_ms']:.3f} ms · {b['tokens_per_s']:,.0f} tok/s · "
      f"bottleneck = {b['bottleneck']} ({b['gemm_bound']}-bound GEMM)")
print(f"  spare compute: {b['spare_compute']:.0%} of the GPU's math capacity is idle\n")
print(f"{'optimization':<18}{'tok/s':>11}{'gain':>8}   new bottleneck")
print("-" * 58)
for name, fn in OPTS.items():
    r = evaluate(fn(base))
    print(f"{name:<18}{r['tokens_per_s']:>11,.0f}{r['tokens_per_s']/b['tokens_per_s']:>7.2f}x"
          f"   {r['bottleneck']}")

## Part 2 · The interaction matrix, computed

Now the central question. For each **pair** of optimizations, compare what you get together against
what the two headline numbers would predict:

$$\text{synergy}(A,B) = \frac{\text{gain}(A{+}B)}{\text{gain}(A) \times \text{gain}(B)}$$

- **≈ 1.0** → independent: they cut different slices, gains genuinely multiply
- **< 1.0** → antagonistic: they compete for the same slice; you paid twice for one win
- **> 1.0** → synergistic: one *unlocks* the other

Crucially this is **derived from the model**, not asserted — so you can change the workload and watch
the matrix change with it.

In [ ]:
def gain(cfg_fns, base=None):
    base = base or base_config()
    cfg = base
    for fn in cfg_fns:
        cfg = fn(cfg)
    return evaluate(cfg)["tokens_per_s"] / evaluate(base)["tokens_per_s"]

# Some optimizations are alternatives, not additions - a model has ONE weight format.
EXCLUSIVE = [{"int4 weights", "fp8 weights"}]

def interaction_matrix(base):
    names = list(OPTS)
    solo = {n: gain([OPTS[n]], base) for n in names}
    cells = []
    for i, a in enumerate(names):
        for j, bname in enumerate(names):
            if i >= j:
                continue
            if {a, bname} in EXCLUSIVE:
                continue                       # mutually exclusive: not an interaction
            both = gain([OPTS[a], OPTS[bname]], base)
            syn = both / (solo[a] * solo[bname])
            cells.append({"a": a, "b": bname, "solo_a": solo[a], "solo_b": solo[bname],
                          "both": both, "synergy": syn})
    return solo, cells

solo, cells = interaction_matrix(base_config())
print(f"baseline: Llama-3.1-8B, H100, batch 32, 2k ctx\n")
print(f"{'A':<16}{'B':<16}{'A alone':>9}{'B alone':>9}{'expected':>10}{'actual':>9}{'synergy':>9}")
print("-" * 80)
for c in sorted(cells, key=lambda c: c["synergy"]):
    expected = c["solo_a"] * c["solo_b"]
    flag = ("  ANTAGONISTIC" if c["synergy"] < 0.9 else
            "  synergistic" if c["synergy"] > 1.1 else "")
    print(f"{c['a']:<16}{c['b']:<16}{c['solo_a']:>8.2f}x{c['solo_b']:>8.2f}x"
          f"{expected:>9.2f}x{c['both']:>8.2f}x{c['synergy']:>8.2f}{flag}")

print("\n(int4 and fp8 weights are omitted as a pair: they are alternatives, not additions -")
print(" a model has exactly one weight format. The search in Part 5 enforces the same rule.)")
worst = min(cells, key=lambda c: c["synergy"])
print(f"\nWorst pair: {worst['a']} + {worst['b']} - you would predict "
      f"{worst['solo_a']*worst['solo_b']:.2f}x and get {worst['both']:.2f}x.")
print("Both spend the same currency: spare compute. Once one has consumed it, the other")
print("has nothing left to convert into tokens.")

In [ ]:
# Heatmap of the interaction matrix.
names = list(OPTS)
grid_cells = []
for c in cells:
    grid_cells.append({"a": c["a"], "b": c["b"], "synergy": round(c["synergy"], 3),
                       "both": round(c["both"], 2),
                       "expected": round(c["solo_a"] * c["solo_b"], 2)})
    grid_cells.append({"a": c["b"], "b": c["a"], "synergy": round(c["synergy"], 3),
                       "both": round(c["both"], 2),
                       "expected": round(c["solo_a"] * c["solo_b"], 2)})

JS = r'''
const names = data.names;
const M = {top: 96, right: 20, bottom: 20, left: 132};
const size = Math.min(52, (W - M.left - M.right) / names.length);
const svg = root.append("svg").attr("width", W)
    .attr("height", M.top + names.length*size + M.bottom)
    .append("g").attr("transform",`translate(${M.left},${M.top})`);
const color = d3.scaleLinear().domain([0.6, 1.0, 1.25])
    .range(["#c62828", "#eceff1", "#2e7d32"]).clamp(true);
const idx = n => names.indexOf(n);

svg.selectAll("c").data(data.cells).join("rect")
   .attr("x", d=>idx(d.b)*size+1).attr("y", d=>idx(d.a)*size+1)
   .attr("width", size-2).attr("height", size-2).attr("rx",3)
   .attr("fill", d=>color(d.synergy))
   .append("title").text(d=>`${d.a} + ${d.b}\nexpected ${d.expected}x, actual ${d.both}x\nsynergy ${d.synergy}`);
svg.selectAll("t").data(data.cells).join("text")
   .attr("x", d=>idx(d.b)*size+size/2).attr("y", d=>idx(d.a)*size+size/2+4)
   .attr("text-anchor","middle").style("font-size","10.5px")
   .style("fill", d=>Math.abs(d.synergy-1)>0.22 ? "white" : "#37474f")
   .text(d=>d.synergy.toFixed(2));
// diagonal
svg.selectAll("d").data(names).join("rect")
   .attr("x",(d,i)=>i*size+1).attr("y",(d,i)=>i*size+1)
   .attr("width",size-2).attr("height",size-2).attr("fill","#fafafa").attr("rx",3);

svg.selectAll("rl").data(names).join("text")
   .attr("x",-8).attr("y",(d,i)=>i*size+size/2+4).attr("text-anchor","end")
   .style("font-size","11px").text(d=>d);
svg.selectAll("cl").data(names).join("text")
   .attr("x",(d,i)=>i*size+size/2).attr("y",-8)
   .attr("transform",(d,i)=>`rotate(-42 ${i*size+size/2} -8)`)
   .style("font-size","11px").text(d=>d);
svg.append("text").attr("x",0).attr("y",-72).style("font-size","11.5px").style("fill","#546e7a")
   .text("red < 0.9 = they fight · grey ≈ 1.0 = independent · green > 1.1 = one unlocks the other");
'''
show_d3(JS, {"names": names, "cells": grid_cells}, height=len(names) * 52 + 130)

**How to read it.** Every red cell is money you would waste by buying both optimizations expecting
their headline numbers. Every grey cell is a genuine multiplication. The pattern isn't arbitrary:

- **Anything that converts spare compute fights anything else that does.** `speculation`, `2× batch`
  — they draw from one pool.
- **Anything that frees memory bandwidth helps things that were bandwidth-starved.** `int4 weights`
  and `fp8 KV` cut different byte streams, so they mostly stack.
- **`KV eviction` and `fp8 KV` both attack the attention slice** — do both and the second one is
  cutting an already-small slice.

## Part 3 · Bottleneck migration

The reason for all of this in one picture: as you stack optimizations, **the ceiling moves**. Watch
where the time goes at each step of a stack.

In [ ]:
STACK = [("baseline", []),
         ("+ graph capture", [opt_graphs]),
         ("+ int4 weights", [opt_graphs, opt_int4]),
         ("+ fp8 KV", [opt_graphs, opt_int4, opt_fp8kv]),
         ("+ 2x batch", [opt_graphs, opt_int4, opt_fp8kv, opt_batch2]),
         ("+ speculation", [opt_graphs, opt_int4, opt_fp8kv, opt_batch2, opt_spec])]

# Start from an UNOPTIMIZED baseline (graphs off), interactive-ish, where spare compute exists.
start = base_config(batch=8); start["graphs"] = False
b0 = evaluate(start)
print(f"start: batch 8, Llama-3.1-8B, H100 · {b0['tokens_per_s']:,.0f} tok/s\n")
print(f"{'stack':<20}{'tok/s':>10}{'cumulative':>12}{'marginal':>10}   bottleneck   spare compute")
print("-" * 86)
migration, prev = [], b0["tokens_per_s"]
for label, fns in STACK:
    cfg = start
    for fn in fns:
        cfg = fn(cfg)
    r = evaluate(cfg)
    migration.append({"label": label, "tps": round(r["tokens_per_s"]),
                      "bottleneck": r["bottleneck"], "spare": round(r["spare_compute"], 3),
                      **{k: round(v * 1000, 5) for k, v in r["parts"].items()}})
    print(f"{label:<20}{r['tokens_per_s']:>10,.0f}{r['tokens_per_s']/b0['tokens_per_s']:>11.2f}x"
          f"{r['tokens_per_s']/prev:>9.2f}x   {r['bottleneck']:<12} {r['spare_compute']:>6.0%}")
    prev = r["tokens_per_s"]

print("\nWatch the last two columns together. Early on, GEMMs dominate and spare compute is high,")
print("so each optimization pays well. By the end the bottleneck has migrated and spare compute")
print("is gone - which is exactly why speculation, a genuinely powerful technique, adds little")
print("HERE while it would have added a lot if applied FIRST.")
print("\nOrdering is not a detail. It changes what each step is worth.")

In [ ]:
JS = r'''
const SLICES = ["GEMMs","attention","communication","sampling","launch","draft"];
const COLORS = {"GEMMs":"#1976d2","attention":"#e53935","communication":"#8e24aa",
                "sampling":"#43a047","launch":"#fb8c00","draft":"#6d4c41"};
const M = {top: 22, right: 132, bottom: 74, left: 60};
const iw = W - M.left - M.right, ih = H - M.top - M.bottom;
const svg = root.append("svg").attr("width",W).attr("height",H).append("g")
    .attr("transform",`translate(${M.left},${M.top})`);
const x = d3.scaleBand().domain(data.map(d=>d.label)).range([0,iw]).padding(0.2);
const y = d3.scaleLinear().domain([0, d3.max(data, d=>d3.sum(SLICES, s=>d[s]||0))*1.12]).range([ih,0]);
svg.append("g").attr("transform",`translate(0,${ih})`).call(d3.axisBottom(x))
   .selectAll("text").attr("transform","rotate(-18)").style("text-anchor","end").style("font-size","10.5px");
svg.append("g").call(d3.axisLeft(y).ticks(5));
svg.append("text").attr("transform","rotate(-90)").attr("x",-ih/2).attr("y",-44)
   .attr("text-anchor","middle").style("font-size","12px").text("decode step time (ms)");
const stack = d3.stack().keys(SLICES).value((d,k)=>d[k]||0);
svg.selectAll("layer").data(stack(data)).join("g").attr("fill",d=>COLORS[d.key])
   .selectAll("rect").data(d=>d.map(v=>({...v,key:d.key}))).join("rect")
   .attr("x",d=>x(d.data.label)).attr("width",x.bandwidth())
   .attr("y",d=>y(d[1])).attr("height",d=>Math.max(0,y(d[0])-y(d[1])))
   .append("title").text(d=>`${d.key}: ${(d[1]-d[0]).toFixed(3)} ms`);
svg.selectAll("bl").data(data).join("text")
   .attr("x",d=>x(d.label)+x.bandwidth()/2).attr("y",d=>y(d3.sum(SLICES,s=>d[s]||0))-6)
   .attr("text-anchor","middle").style("font-size","10px").style("font-weight",600)
   .style("fill","#37474f").text(d=>d.bottleneck);
SLICES.forEach((s,i)=>{
  svg.append("rect").attr("x",iw+12).attr("y",i*17).attr("width",11).attr("height",11)
     .attr("rx",2).attr("fill",COLORS[s]);
  svg.append("text").attr("x",iw+28).attr("y",i*17+10).style("font-size","10.5px").text(s);
});
'''
show_d3(JS, migration, height=380)

**The label above each bar is the bottleneck at that stage.** It moves. That migration is the
mechanism behind every red cell in Part 2's matrix.

## Part 4 · The waterfall: expected vs actual

Put the naive multiplication next to reality.

In [ ]:
order = ["int4 weights", "fp8 KV cache", "2x batch", "speculation"]
start = base_config(batch=8)
b0 = evaluate(start)["tokens_per_s"]

cum_fns, waterfall, naive = [], [], 1.0
for name in order:
    solo_gain = gain([OPTS[name]], start)
    naive *= solo_gain
    cum_fns.append(OPTS[name])
    actual = gain(cum_fns, start)
    waterfall.append({"opt": name, "solo": round(solo_gain, 3),
                      "naive_cumulative": round(naive, 3), "actual_cumulative": round(actual, 3)})

print(f"{'applied':<18}{'solo gain':>11}{'if they multiplied':>21}{'actually':>11}{'shortfall':>12}")
print("-" * 76)
for w in waterfall:
    short = 1 - w["actual_cumulative"] / w["naive_cumulative"]
    print(f"{w['opt']:<18}{w['solo']:>10.2f}x{w['naive_cumulative']:>20.2f}x"
          f"{w['actual_cumulative']:>10.2f}x{short:>11.0%}")

final = waterfall[-1]
print(f"\nThe four headline numbers multiply to {final['naive_cumulative']:.1f}x.")
print(f"You actually get {final['actual_cumulative']:.1f}x - "
      f"{1-final['actual_cumulative']/final['naive_cumulative']:.0%} less than the brochure.")
print("\nTwo things worth noticing in that table, because they pull in opposite directions:")
print("  - NEGATIVE shortfall early: an optimization can be worth MORE in a stack than alone.")
print("    fp8 KV looks unimpressive by itself (the GEMM slice was hiding attention), but once")
print("    int4 has shrunk the GEMMs, attention is exposed and cutting it suddenly pays.")
print("  - LARGE shortfall late: by the time speculation is applied, the spare compute it")
print("    needs has already been spent by everything before it.")
print("\nSo the headline numbers are not lies - they are measurements taken against an")
print("unoptimized baseline, which is precisely the situation you are no longer in after")
print("your first optimization.")

JS = r'''
const M = {top: 20, right: 24, bottom: 66, left: 62};
const iw = W - M.left - M.right, ih = H - M.top - M.bottom;
const svg = root.append("svg").attr("width",W).attr("height",H).append("g")
    .attr("transform",`translate(${M.left},${M.top})`);
const x = d3.scaleBand().domain(data.map(d=>d.opt)).range([0,iw]).padding(0.28);
const y = d3.scaleLinear().domain([0, d3.max(data,d=>d.naive_cumulative)*1.12]).range([ih,0]);
svg.append("g").attr("transform",`translate(0,${ih})`).call(d3.axisBottom(x))
   .selectAll("text").attr("transform","rotate(-16)").style("text-anchor","end").style("font-size","11px");
svg.append("g").call(d3.axisLeft(y).tickFormat(d=>d+"x"));
svg.append("text").attr("transform","rotate(-90)").attr("x",-ih/2).attr("y",-46)
   .attr("text-anchor","middle").style("font-size","12px").text("cumulative speedup");
const sub = x.bandwidth()/2;
svg.selectAll("n").data(data).join("rect")
   .attr("x",d=>x(d.opt)).attr("width",sub-2).attr("y",d=>y(d.naive_cumulative))
   .attr("height",d=>ih-y(d.naive_cumulative)).attr("fill","#cfd8dc").attr("rx",2)
   .append("title").text(d=>`if gains multiplied: ${d.naive_cumulative}x`);
svg.selectAll("a").data(data).join("rect")
   .attr("x",d=>x(d.opt)+sub).attr("width",sub-2).attr("y",d=>y(d.actual_cumulative))
   .attr("height",d=>ih-y(d.actual_cumulative)).attr("fill","#2e7d32").attr("rx",2)
   .append("title").text(d=>`actual: ${d.actual_cumulative}x`);
svg.selectAll("l").data(data).join("text")
   .attr("x",d=>x(d.opt)+sub+sub/2).attr("y",d=>y(d.actual_cumulative)-5)
   .attr("text-anchor","middle").style("font-size","10.5px").style("fill","#2e7d32")
   .text(d=>d.actual_cumulative.toFixed(2)+"x");
svg.append("text").attr("x",4).attr("y",12).style("font-size","11.5px").style("fill","#546e7a")
   .text("grey = what the headline numbers predict · green = what you actually get");
'''
show_d3(JS, waterfall, height=330)

## Part 5 · Searching for the best stack

If gains don't multiply, you can't pick optimizations greedily by headline number. So **search**.
Every subset gets scored — with a budget for the two things that actually constrain teams:
**accuracy risk** and **engineering effort**.

In [ ]:
# `risk` here means ACCURACY risk specifically - the chance of changing what the model outputs.
# Latency/capacity trade-offs are real but belong in the SLO analysis (nb 27), not here.
COST = {
    "int4 weights":  dict(risk=2, effort=1, note="lossy: verify quality at your context length (nb 23)"),
    "fp8 weights":   dict(risk=1, effort=1, note="mildly lossy; needs FP8 hardware (nb 32)"),
    "fp8 KV cache":  dict(risk=1, effort=1, note="lossy, and worse at long context (nb 34)"),
    "2x batch":      dict(risk=0, effort=1, note="lossless; costs TPOT - watch the knee (nb 27)"),
    "speculation":   dict(risk=0, effort=3, note="PROVABLY lossless (nb 24); costs a draft model"),
    "TP=2":          dict(risk=0, effort=3, note="lossless; doubles GPUs per replica (nb 29)"),
    "KV eviction":   dict(risk=4, effort=3, note="very lossy; wrong for retrieval tasks (nb 34)"),
}
print("Accuracy risk vs effort for each optimization (risk 0 = provably output-preserving):\n")
for name, c in COST.items():
    print(f"  {name:<16}risk {c['risk']}  effort {c['effort']}   {c['note']}")
print("\nNote the two zero-risk-but-not-free entries: batching and speculation change")
print("NOTHING about the output distribution. They cost latency and engineering, not quality.\n")

def search(base, risk_budget=99, effort_budget=99, gpu_budget=1):
    names = list(OPTS)
    results = []
    for mask in range(1 << len(names)):
        chosen = [names[i] for i in range(len(names)) if mask >> i & 1]
        if "int4 weights" in chosen and "fp8 weights" in chosen:
            continue                                  # mutually exclusive: one weight format
        risk = sum(COST[c]["risk"] for c in chosen)
        effort = sum(COST[c]["effort"] for c in chosen)
        gpus = 2 if "TP=2" in chosen else 1
        if risk > risk_budget or effort > effort_budget or gpus > gpu_budget:
            continue
        g = gain([OPTS[c] for c in chosen], base)
        results.append({"stack": chosen, "gain": g, "risk": risk, "effort": effort,
                        "gpus": gpus, "per_gpu": g / gpus})
    return sorted(results, key=lambda r: -r["per_gpu"])

start = base_config(batch=8)
print("Llama-3.1-8B, H100, batch 8, 2k ctx — best stacks under different budgets:\n")
for label, kw in [("no constraints", dict(gpu_budget=2)),
                  ("single GPU", dict(gpu_budget=1)),
                  ("low risk (<=2)", dict(risk_budget=2, gpu_budget=1)),
                  ("low effort (<=2)", dict(effort_budget=2, gpu_budget=1)),
                  ("zero accuracy risk", dict(risk_budget=0, gpu_budget=1))]:
    best = search(start, **kw)[0]
    print(f"{label:<22}{best['per_gpu']:>6.2f}x/GPU  risk {best['risk']}  effort {best['effort']}")
    print(f"{'':<22}{' + '.join(best['stack']) or '(do nothing)'}")

print("\nNotice how different the answers are. 'Best' is not a property of an optimization -")
print("it is a property of an optimization AND your risk tolerance AND your GPU budget.")

print("\n\nThe effort frontier - best achievable gain for each engineering budget:\n")
all_r = search(start, gpu_budget=1)
ceiling = max(r["per_gpu"] for r in all_r)
print(f"{'effort':>7}{'best gain':>11}{'% of ceiling':>14}   stack")
print("-" * 78)
prev_best = 0
for eff in range(0, 10):
    at = [r for r in all_r if r["effort"] <= eff]
    if not at:
        continue
    best = max(at, key=lambda r: r["per_gpu"])
    if best["per_gpu"] <= prev_best + 1e-9:
        continue                                  # nothing new unlocked at this budget
    prev_best = best["per_gpu"]
    print(f"{eff:>7}{best['per_gpu']:>10.2f}x{best['per_gpu']/ceiling:>13.0%}   "
          f"{' + '.join(best['stack']) or '(nothing)'}")

print("\nThe steep part of that frontier is where you should live: the first 1-2 units of")
print("effort typically buy most of the achievable gain. Everything past the knee is")
print("engineering you are spending for a diminishing slice - which is the same shape as")
print("the diminishing returns inside a single stack (Part 4), one level up.")

## Part 6 · Interactions the arithmetic can't see

The model above captures *performance* coupling. Some of the most important interactions are
structural, and no throughput number will reveal them. These are the ones that cause outages and
quality regressions:

| Pair | Interaction | Why |
|---|---|---|
| **KV eviction × prefix caching** | 💥 **mutually destructive** | Eviction throws away KV blocks; prefix caching's whole value is *keeping* them for reuse. Evict aggressively and your hit rate collapses (nb 22, 34) |
| **Prefix caching × per-request randomness** | 💥 destructive | A timestamp or request-ID at the top of a prompt invalidates every cached block (nb 36) |
| **Speculation × large batch** | ⚠️ self-cancelling | Speculation needs spare compute; batching consumes it. Engines often auto-disable speculation under load (nb 24) |
| **Multi-LoRA × speculation** | ⚠️ overhead stacking | Both add per-step kernel launches — the slice notebook 38 showed is fixed-cost (nb 30) |
| **Quantization × long context** | ⚠️ compounding error | Low-precision KV accumulates error over more attended tokens; evaluate at your **target** context, not 2k (nb 34) |
| **TP × speculation** | ✅ synergistic | Speculation emits several tokens per verify pass, so you pay TP's all-reduce **less often per token** (nb 29) |
| **Prefix caching × prefix-aware routing** | ✅ required pairing | A cache the router doesn't respect is a cache you don't have (nb 29, 36) |
| **Chunked prefill × speculation** | ⚠️ latency coupling | Both reshape decode step time; tune them together or you'll chase your own tail (nb 34) |

**The first row deserves emphasis** because both techniques are individually recommended for
long-context serving, and teams enable both: aggressive KV eviction and prefix caching are working
against each other by construction. If your workload is repetitive (agents, chat, RAG), **prefix
caching is worth far more** — keep the blocks (nb 36).

## Part 7 · Recipes

Derived by running Part 5's search against each workload shape:

In [ ]:
WORKLOADS = [
    ("interactive chat (1 user)",  base_config(batch=1,  ctx=4096)),
    ("high-throughput API",        base_config(batch=128, ctx=2048)),
    ("RAG / doc Q&A",              base_config(batch=16, ctx=16384)),
    ("long-context agent",         base_config(batch=8,  ctx=32768)),
    ("small model, huge scale",    base_config(model="Qwen2.5-0.5B", batch=256, ctx=1024)),
    ("70B on one node",            base_config(model="Llama-3.1-70B", batch=32, ctx=4096, gpu="MI300X")),
]
print(f"{'workload':<28}{'bottleneck':<14}{'best low-risk stack (<=2), 1 GPU':<44}{'gain':>7}")
print("-" * 96)
for label, cfg in WORKLOADS:
    ev = evaluate(cfg)
    best = search(cfg, risk_budget=2, gpu_budget=1)[0]
    stack = " + ".join(best["stack"]) or "(nothing helps much)"
    print(f"{label:<28}{ev['bottleneck']:<14}{stack:<44}{best['gain']:>6.2f}x")

print("\nRead down the bottleneck column: it is different for almost every workload, which is")
print("why the recommended stack is different too. Anyone who gives you a single ordered list")
print("of 'the top 5 LLM serving optimizations' is describing their workload, not yours.")

### The one-page decision sheet

```
 1. MEASURE the decode step (nb 38 Part 6) and the logs (nb 26).
    → you now know your biggest slice and whether you have spare compute.

 2. FREE WINS FIRST — no accuracy risk, no new hardware:
    • graph capture on                      (kills the fixed overhead slice)
    • prompt layout: static → volatile      (unlocks prefix caching, nb 36)
    • prefix-aware routing                  (makes that cache real, nb 29)
    • right-size --max-model-len            (buys back KV pool, nb 26)

 3. MATCH THE TECHNIQUE TO THE SLICE (nb 38 Part 5):
    GEMM slice biggest  → quantize weights (int4/FP8)
    attention biggest   → fp8 KV, then consider eviction/SWA (nb 34)
    overhead biggest    → graph capture; reduce kernel count
    spare compute       → speculation (nb 24) OR bigger batch — NOT both

 4. RE-MEASURE. The bottleneck has moved. Step 3's answer has changed.

 5. ONLY THEN spend money: bigger GPU, more replicas, TP (nb 29, 31, 33).
```

**The loop in step 4 is the entire point of this notebook.** Optimization is not a checklist you
execute once; it is a search where each move changes the board.

## Recap

1. **Gains don't multiply.** Four optimizations with headline numbers multiplying to a large factor
   routinely deliver a fraction of it, because each works on a smaller problem than the last.
2. **The interaction matrix is computable**, not folklore — and it changes with your workload.
3. **Optimizations that spend the same currency fight.** Spare compute is the scarcest currency:
   speculation and batching both spend it.
4. **The bottleneck migrates as you stack.** Where it lands determines what's worth doing next.
5. **Order matters** — the same set of optimizations is worth different amounts depending on the
   sequence you apply them in.
6. **Some pairs are structurally destructive** and no throughput number reveals it. KV eviction and
   prefix caching are the pair to watch.
7. **Search under a risk and effort budget**, then ship the cheapest stack that reaches ~80% of the
   ceiling.

### Further reading
- [Amdahl's law](https://en.wikipedia.org/wiki/Amdahl%27s_law) — the 1967 version of this entire notebook
- [Roofline model](https://dl.acm.org/doi/10.1145/1498765.1498785) — where "which resource is scarce" comes from
- The optimizations themselves: nb [22](./vLLM_High_Throughput_Serving.ipynb) · [23](./Quantized_Serving_Showdown.ipynb) · [24](./Speculative_Decoding_Advanced_Serving.ipynb) · [34](./LongContext_KV_Compression_Serving.ipynb) · [36](./RAG_Agent_Serving_Patterns.ipynb)
- The measurement that grounds it: [38](./Anatomy_Of_A_Decode_Step.ipynb) · [27](./Serving_Benchmark_Capacity_Planning.ipynb)

🏁 **End of the serving track.** [Back to the learning path](README.md).